# Crop Yield Trend Summary — SQL

SQL-cell equivalent of `crop_yield_trend_summary.ipynb`, using `%%sql`
cells (pivot via conditional aggregation, trend via `LAG`) instead of the
DataFrame API. Same result: reads `agriculture.crop_statistics` and
persists `agriculture.crop_yield_trend_summary_v1`.

Requires the bronze table to already exist — run
`data-generators/usda_crop_statistics_ingest.ipynb` first.

`%%sql` here follows Databricks-style notebook cell magic. If running
outside a platform that provides it natively, install and load
`sparksql-magic` first: `pip install sparksql-magic` then
`%load_ext sparksql_magic` (bound to the `spark` session created below).

In [ ]:
from pyspark.sql import SparkSession

spark = SparkSession.builder.getOrCreate()

## Transform + persist: yield trend by state/commodity

Cleans `VALUE` (strips commas, nulls out `(D)`/`(Z)`/`(NA)` sentinels via
a regex check), pivots stat categories into columns with conditional
aggregation, then computes the change between census years via `LAG`.

In [ ]:
%%sql
DROP TABLE IF EXISTS agriculture.crop_yield_trend_summary_v1;

CREATE TABLE agriculture.crop_yield_trend_summary_v1 AS
WITH cleaned AS (
    SELECT
        STATE_ALPHA AS state,
        COMMODITY_DESC AS commodity,
        YEAR AS census_year,
        STATISTICCAT_DESC,
        CASE
            WHEN REGEXP_REPLACE(VALUE, ',', '') RLIKE '^[0-9.]+$'
            THEN CAST(REGEXP_REPLACE(VALUE, ',', '') AS DOUBLE)
        END AS value_clean
    FROM agriculture.crop_statistics
),
pivoted AS (
    SELECT
        state,
        commodity,
        census_year,
        MAX(CASE WHEN STATISTICCAT_DESC = 'AREA HARVESTED' THEN value_clean END) AS area_harvested_acres,
        MAX(CASE WHEN STATISTICCAT_DESC = 'PRODUCTION' THEN value_clean END) AS production,
        MAX(CASE WHEN STATISTICCAT_DESC = 'YIELD' THEN value_clean END) AS yield_per_acre
    FROM cleaned
    GROUP BY state, commodity, census_year
)
SELECT
    state,
    commodity,
    census_year,
    area_harvested_acres,
    production,
    yield_per_acre,
    LAG(yield_per_acre) OVER (PARTITION BY state, commodity ORDER BY census_year) AS prior_census_yield,
    ROUND(
        (yield_per_acre - LAG(yield_per_acre) OVER (PARTITION BY state, commodity ORDER BY census_year)) * 100.0
        / LAG(yield_per_acre) OVER (PARTITION BY state, commodity ORDER BY census_year),
        1
    ) AS yield_change_pct
FROM pivoted
ORDER BY state, commodity, census_year

## Verify

In [ ]:
%%sql
SELECT COUNT(*) AS total_rows FROM agriculture.crop_yield_trend_summary_v1

In [ ]:
%%sql
SELECT * FROM agriculture.crop_yield_trend_summary_v1
WHERE yield_change_pct IS NOT NULL
ORDER BY yield_change_pct DESC
LIMIT 10